# GraphRAG

## Import packages

In [15]:
import sys
sys.path.append('..')
sys.path.append('../neurorag')
sys.path.append('../neurorag/chains')

import os
import pandas as pd
from tqdm import tqdm
from pathlib import Path
import json
from dotenv import load_dotenv
from getpass import getpass

from neurorag.neurorag import NeuroRAG

from metrics import (
  embeddings_cosine_sim_metric,
  bleu_metric,
  rogue_l_metric,
  rogue_1_metric,
  factscore_metric,
  summac_zs_metric,
  summac_conv_metric,
)

## Disable warnings

In [16]:
import warnings
warnings.filterwarnings('ignore')

## Setup environment variables

You have to define the following environment variables in the `.env` file, terminal environment, or input field within this Jupyter notebook.

## Import packages

In [17]:
env_variables = [
  'TAVILY_API_KEY',
  'ENTREZ_EMAIL',
  'OPENROUTER_API_KEY',
  'CHROMA_API_KEY',
  'CHROMA_TENANT',
  'CHROMA_DATABASE',
  'CHROMA_COLLECTION_NAME',
]

load_dotenv()

for key in env_variables:
  value = os.getenv(key)

  if value is None:
    value = getpass(key)

  os.environ[key] = value

## Build model

In [18]:
app = NeuroRAG(debug=True, use_flare=False)
app.compile()

## Evaluate RAG

### Load QA dataset

In [19]:
mediqa_df = pd.read_csv('../datasets/pubmed_summary_qa.csv')
mediqa_df

,question,answer
0,What is the significance of the inferior prefr...,"The inferior prefrontal cortex, including area..."
1,How do circadian activity rhythms affect cogni...,Consistent circadian activity rhythms are asso...
2,How do executive function deficits affect read...,Executive function deficits can affect reading...
3,How does brain activity differ between highly ...,Highly hypnotizable individuals tend to exhibi...
4,What is the role of predictive neural activity...,Predictive neural activity plays a central rol...
5,How is serotonin signalling related to aggress...,Serotonin signalling has been implicated in th...
6,What brain regions are involved in motor urgency?,"The cerebellum, sensorimotor cortex, and prefr..."
7,How does bilingualism affect cognitive develop...,Bilingualism has been shown to influence cogni...
8,What is the role of the insula in major depres...,The insula is thought to contribute to the pat...
9,How does bilingualism affect cognitive process...,Bilingualism can influence cognitive processin...


### Load cached RAGs responses

In [20]:
cache_path = Path('cache.json')

if not os.path.exists(cache_path):
  data = {}
  with open(cache_path, 'w') as file:
    json.dump(data, file)

with open(cache_path, 'r') as f:
  cache = json.load(f)

CACHE_KEY = 'text-to-text-neurorag-evaluation'

if CACHE_KEY not in cache:
  cache[CACHE_KEY] = {}

len(cache.keys())

3

In [21]:
questions = list(mediqa_df['question'].tolist())
expected_answers = list(mediqa_df['answer'].tolist())
predicted_answers = []

for index, question in tqdm(enumerate(questions)):
  if question not in cache[CACHE_KEY]:
    cache[CACHE_KEY][question] = app.invoke(question)['generation']

  predicted_answers.append(cache[CACHE_KEY][question])

  with open(cache_path, 'w') as f:
    json.dump(cache, f)

cos_score = embeddings_cosine_sim_metric(expected_answers, predicted_answers)
bleu_score = bleu_metric(expected_answers, predicted_answers)
rogue_1_score = rogue_1_metric(expected_answers, predicted_answers)
rogue_l_score = rogue_l_metric(expected_answers, predicted_answers)
# factscore_score = factscore_metric(expected_answers, predicted_answers)
# summac_zs_score = summac_zs_metric(expected_answers, predicted_answers)
# summac_conv_score = summac_conv_metric(expected_answers, predicted_answers)

# cos_score, bleu_score, rogue_1_score, rogue_l_score, factscore_score, summac_zs_score, summac_conv_score
cos_score, bleu_score, rogue_1_score, rogue_l_score

8it [00:00, 10.98it/s]

[2026-02-25 19:47:12.791] ---GENERATE STEP-BACK QUERY---
[2026-02-25 19:47:16.962] ---GENERATE SUBQUERIES---
[2026-02-25 19:47:21.914] ---DETERMINE SPECIALIZED SOURCES---
[2026-02-25 19:47:27.981] ---SELECTED SOURCES: ['vectorstore', 'pubmed']---
[2026-02-25 19:47:27.981] ---ROUTE QUESTION---
[2026-02-25 19:47:27.982] ---GENERATE HYDE DOCUMENTS---


8it [00:18, 10.98it/s]

[2026-02-25 19:47:31.216][2026-02-25 19:47:31.217] ---RETRIEVE FROM VECTOR STORE---
 ---RETRIEVE FROM PUBMED---
[2026-02-25 19:47:31.825] pub_med_retriever_node HTTP Error 429: Too Many Requests
Too Many Requests, waiting for 0.20 seconds...
Too Many Requests, waiting for 0.20 seconds...
Too Many Requests, waiting for 0.40 seconds...
Too Many Requests, waiting for 1.60 seconds...
Too Many Requests, waiting for 1.60 seconds...
Too Many Requests, waiting for 6.40 seconds...
[2026-02-25 19:47:43.738] ---GRADE DOCUMENTs---
[2026-02-25 19:47:43.738] ---AFTER EXACT DEDUPLICATION: 13 documents---
[2026-02-25 19:47:43.741] ---BM25 TOP CANDIDATES: 10 documents---
[2026-02-25 19:47:50.508] ---FINAL DOCUMENTS NUMBER: 3---
[2026-02-25 19:47:50.509] ---ASSESS GRADED DOCUMENTS---
[2026-02-25 19:47:50.509] ---DECISION: GENERATE---
[2026-02-25 19:47:50.509] ---GENERATE---
[2026-02-25 19:48:18.174] ---GRADE GENERATION---
[2026-02-25 19:48:19.866] ---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---


10it [01:10, 12.58s/it]

...

[2026-02-25 20:07:06.814] ---DECISION: GENERATION ADDRESSES QUESTION---
[2026-02-25 20:07:06.934] ---GENERATE STEP-BACK QUERY---
[2026-02-25 20:07:08.315] ---GENERATE SUBQUERIES---
[2026-02-25 20:07:08.644] ---DETERMINE SPECIALIZED SOURCES---
[2026-02-25 20:07:11.105] ---SELECTED SOURCES: ['vectorstore', 'pubmed']---
[2026-02-25 20:07:11.106] ---ROUTE QUESTION---
[2026-02-25 20:07:11.106] ---GENERATE HYDE DOCUMENTS---
[2026-02-25 20:07:14.883][2026-02-25 20:07:14.884] ---RETRIEVE FROM VECTOR STORE---
 ---RETRIEVE FROM PUBMED---
Too Many Requests, waiting for 819.20 seconds...
Too Many Requests, waiting for 819.20 seconds...
Too Many Requests, waiting for 819.20 seconds...
Too Many Requests, waiting for 819.20 seconds...
[2026-02-25 20:09:14.888] pub_med_retriever_node timed out
[2026-02-25 20:09:14.889] ---GRADE DOCUMENTs---
[2026-02-25 20:09:14.889] ---AFTER EXACT DEDUPLICATION: 11 documents---
[2026-02-25 20:09:14.891] ---BM25 TOP CANDIDATES: 10 documents---
[2026-02-25 20:09:18.533]

20it [22:32, 67.62s/it] 

[2026-02-25 20:09:44.253] ---GRADE GENERATION---


(0.7608654934129035,
 0.015918986722471978,
 0.2819620515953176,
 0.20790083151527464)

Too Many Requests, waiting for 6553.60 seconds...
Too Many Requests, waiting for 104857.60 seconds...
Too Many Requests, waiting for 104857.60 seconds...
Too Many Requests, waiting for 104857.60 seconds...
Too Many Requests, waiting for 104857.60 seconds...
